<a href="https://colab.research.google.com/github/Joao-H-Luz/Trabalho-KNN/blob/main/Trabalho_K_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Trabalho KNN
  - Nomes:
      - [João Henrique da Luz](https://github.com/Joao-H-Luz)
      - [Ighor Telles](https://github.com/ighortelles)
      - [Frederico Fragoso Franke](https://github.com/f.franke)


# Bibliotecas
---


In [6]:
import pandas as pd
from sklearn import neighbors
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## Carregar o DF
---

In [18]:
df = pd.read_csv("cardio_v2.csv", sep=";", encoding="latin1")
df.shape

(70000, 13)

In [19]:
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110.0,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140.0,-1,3,1,0,0,1,1
2,2,18857,1,165,64.0,130.0,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150.0,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100.0,60,1,1,0,0,0,0


## Tratamento
---

In [20]:
# ID
df = df.drop(columns=['id'])

# Idade
df["age"] = df["age"].astype(str).str[-2:].astype(int)
df["age"] = 1900 + df["age"]
df.loc[df["age"] <= 1923, "age"] += 100

df["age"] = (2026 - df["age"])

# Altura
df["height"] = df["height"] / 100
df = df[(df["height"] >= 1.32)&(df["height"] <= 2.10)]

# Peso
df = df[(df["weight"] >= 40)&(df["weight"] <= 180)]

# Preção Sistólica
df = df[(df["ap_hi"] >= 90)&(df["ap_hi"] <= 120)]

# Preção Diastólica
df = df[(df["ap_lo"] >= 60)&(df["ap_lo"] <= 80)]

# IMC
df["imc"] = df.weight / (df.height)**2

df = df.round(2)

df.describe().round(2)

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,imc
count,38649.00,38649.00,38649.00,38649.00,38649.00,38649.00,38649.00,38649.00,38649.00,38649.00,38649.0,38649.00,38649.00
mean,52.52,1.33,1.65,70.97,115.74,76.24,1.23,1.17,0.08,0.04,0.8,0.31,26.23
std,28.83,0.47,0.08,12.53,7.21,6.03,0.55,0.51,0.27,0.21,0.4,0.46,4.51
min,3.00,1.00,1.32,40.00,90.00,60.00,1.00,1.00,0.00,0.00,0.0,0.00,14.53
25%,28.00,1.00,1.60,63.00,110.00,70.00,1.00,1.00,0.00,0.00,1.0,0.00,23.37
50%,52.00,1.00,1.65,69.00,120.00,80.00,1.00,1.00,0.00,0.00,1.0,0.00,25.28
75%,78.00,2.00,1.70,78.00,120.00,80.00,1.00,1.00,0.00,0.00,1.0,1.00,28.36
max,102.00,2.00,2.07,178.00,120.00,80.00,3.00,3.00,1.00,1.00,1.0,1.00,68.31


In [63]:
df = df.drop(columns=['height','weight'])

In [64]:
df = df.dropna()
df

,age,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,imc
0,33,2,110.0,80,1,1,0,0,1,0,21.97
4,52,1,100.0,60,1,1,0,0,0,0,23.01
5,12,1,120.0,80,2,2,0,0,0,0,29.38
8,58,1,110.0,70,1,1,0,0,1,0,28.44
9,92,1,110.0,60,1,1,0,0,0,0,25.28
...,...,...,...,...,...,...,...,...,...,...,...
69988,48,1,110.0,70,1,1,0,0,1,0,23.05
69989,13,1,120.0,70,1,1,0,0,1,1,33.67
69990,32,1,110.0,70,1,1,0,0,1,1,25.51
69995,86,2,120.0,80,1,1,1,0,1,0,26.93


## Separar a Base em Treino (80%) e Teste (20%)
---

In [65]:
X = df.drop(columns=['cardio'])
X.head()

,age,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,imc
0,33,2,110.0,80,1,1,0,0,1,21.97
4,52,1,100.0,60,1,1,0,0,0,23.01
5,12,1,120.0,80,2,2,0,0,0,29.38
8,58,1,110.0,70,1,1,0,0,1,28.44
9,92,1,110.0,60,1,1,0,0,0,25.28


In [66]:
y = df["cardio"].values
print(y)

[0 0 0 ... 1 0 0]


In [67]:
X_train, X_aux, y_train, y_aux = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [68]:
X_validation, X_test, y_validation, y_test = train_test_split(
    X_aux, y_aux,
    test_size=0.5,
    random_state=42,
    stratify=y_aux)

## Treinamento
---

In [73]:
# Modelo
clf = neighbors.KNeighborsClassifier(n_neighbors=28)

# Treino
clf.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=28)

In [74]:
# Teste
y_pred = clf.predict(X_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

### Medidas
 ---

In [75]:
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

Accuracy: 0.6846054333764554


In [76]:
acc = accuracy_score(y_validation, y_pred)
print("Accuracy:", acc)

Accuracy: 0.6695989650711514
